# Notebook 11 — Model Diversity

**Goal:** Train architecturally different models on the 38-feature v2 set. Measure Spearman ρ vs LGBM OOF predictions. Target: at least one model pair achieves ρ < 0.90, enabling meaningful stacking in Notebook 13.

**Why this matters:** LGBM + XGBoost had ρ = 0.974 (Notebook 08) — far too correlated to benefit from ensembling. Breaking below ρ = 0.90 requires a genuinely different learning algorithm (bagging vs boosting) or a different inductive bias (dropout-regularised DART).

**Inputs:** `data/processed/train_features_v2.parquet`, `data/processed/test_features_v2.parquet`, `data/processed/fold_assignments.parquet`  
**Outputs:** `data/processed/oof_predictions_nb11.parquet`, `data/processed/test_predictions_nb11.parquet`

**Models trained:**

| Model | Algorithm | Diversity mechanism |
|-------|-----------|--------------------|
| `lgbm_ref` | LightGBM GBDT | Reference baseline (same params as NB 10 Tier 4) |
| `rf` | Random Forest | Bagging + feature subsampling — averaged boundaries |
| `et` | Extra-Trees | Bagging + random splits — maximum variance reduction |
| `dart` | LightGBM DART | Random tree dropout — different prediction surface |
| `cb_plain` | CatBoost Plain | Standard symmetric GBDT — different tree structure |

**Success metric:** Spearman ρ < 0.90 vs `lgbm_ref` OOF for at least one new model, with solo CV AUC ≥ 0.84.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import time
import warnings
import pickle
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print('WARNING: catboost not installed — CatBoost cell will be skipped')

warnings.filterwarnings('ignore')
print(f'LightGBM  : {lgb.__version__}')
print(f'CatBoost  : {"available" if CATBOOST_AVAILABLE else "NOT installed"}')
print('Imports OK')

LightGBM  : 4.6.0
CatBoost  : available
Imports OK


In [2]:
# Robust project root detection — works from workspace root, notebooks/, or Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root  = cwd
data_dir      = project_root / 'data' / 'raw'
processed_dir = project_root / 'data' / 'processed'
models_dir    = project_root / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

# Kaggle path fallback
if not data_dir.exists():
    data_dir = Path('/kaggle/input/playground-series-s6e5')
    processed_dir = Path('/kaggle/working')

print(f'Project root : {project_root}')
print(f'Processed dir: {processed_dir}')

train = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test  = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds = pd.read_parquet(processed_dir / 'fold_assignments.parquet')

# Merge fold assignments into train (not saved in v2 parquet)
train = train.merge(folds, on='id', how='left')

print(f'Train: {train.shape}  |  Test: {test.shape}')
print(f'Fold col present: {"fold" in train.columns}')
print(f'Fold distribution:\n{train["fold"].value_counts().sort_index().to_string()}')

Project root : c:\Repos\predict-f1-pit-stops
Processed dir: c:\Repos\predict-f1-pit-stops\data\processed
Train: (439140, 53)  |  Test: (188165, 51)
Fold col present: True
Fold distribution:
fold
0    88018
1    87444
2    88027
3    87854
4    87797


In [3]:
# ── Feature set: FEATS_T4 from NB 10 (38 features) ─────────────────────────
# Tiers 5 and 6 were dropped (Δ < 0.001). Using the same 38-feature set that
# achieved OOF AUC 0.8987 with LGBM.

BASE_FEATURES = [
    'TyreLife_normalized_by_compound', 'TyreLife_sq',
    'is_wet_tyre', 'compound_ordinal',
    'laps_remaining', 'is_testing_session',
    'Stint', 'Position',
    'Cumulative_Degradation_winsorized', 'Degradation_rate', 'Degradation_acceleration',
    'LapTime_lag1', 'LapTime_lag2', 'LapTime_lag3',
    'LapTime_Delta_lag1', 'LapTime_Delta_lag2', 'LapTime_Delta_lag3',
    'LapTime_rolling_mean_3', 'LapTime_rolling_mean_5',
    'LapTime_rolling_std_3',  'LapTime_rolling_std_5',
    'Degradation_rolling_slope_3', 'Degradation_rolling_slope_5',
    'TyreLife_x_laps_remaining', 'Degradation_x_RaceProgress', 'Position_x_RaceProgress',
]
TARGET_ENC    = ['Race_target_encoded', 'Driver_target_encoded', 'Driver_avg_stint_length']
TIER14_FEATS  = [
    'TyreLife_x_compound_ordinal', 'Stint_x_compound_ordinal', 'TyreLife_x_cmpd_x_laps_rem',  # Tier 1
    'prime_pit_window', 'prime_window_x_compound',                                              # Tier 2
    'abs_position_change', 'pos_change_rolling_std_3',                                          # Tier 3
    'PitStop_lag1', 'laps_to_driver_end',                                                       # Tier 4
]
FEAT_COLS = BASE_FEATURES + TARGET_ENC + TIER14_FEATS  # 38 total

# Verify all non-encoding features exist in parquets
non_enc = BASE_FEATURES + TIER14_FEATS
missing_train = [f for f in non_enc if f not in train.columns]
missing_test  = [f for f in non_enc if f not in test.columns]
assert not missing_train, f'Missing in train: {missing_train}'
assert not missing_test,  f'Missing in test:  {missing_test}'
print(f'Feature set: {len(FEAT_COLS)} features (26 base + 3 target enc + 9 Tier 1-4)')
print(f'Non-null check: train={train[non_enc].isnull().sum().sum()}  test={test[non_enc].isnull().sum().sum()}')


def apply_target_encodings(train_df: pd.DataFrame, val_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute fold-aware target encodings from train_df, apply to val_df.
    Must be called INSIDE each CV fold — never on the full dataset.
    """
    global_mean  = train_df['PitNextLap'].mean()
    race_map     = train_df.groupby('Race')['PitNextLap'].mean()
    driver_map   = train_df.groupby('Driver')['PitNextLap'].mean()
    stint_map    = (
        train_df.groupby(['Driver', 'Race', 'Year', 'Stint'])['TyreLife']
        .max().reset_index().groupby('Driver')['TyreLife'].mean()
    )
    global_stint = stint_map.mean()

    out = val_df.copy()
    out['Race_target_encoded']     = out['Race'].map(race_map).fillna(global_mean)
    out['Driver_target_encoded']   = out['Driver'].map(driver_map).fillna(global_mean)
    out['Driver_avg_stint_length'] = out['Driver'].map(stint_map).fillna(global_stint)
    return out

print('apply_target_encodings defined.')

Feature set: 38 features (26 base + 3 target enc + 9 Tier 1-4)
Non-null check: train=0  test=0
apply_target_encodings defined.


## Generic CV Runner

`run_cv()` handles all model types through three modes:
- **Standard (RF, ET, LGBM-DART):** `m.fit(X_tr, y_tr)` — no eval set, no early stopping
- **LGBM with early stopping:** pass `lgbm_early_stopping=True` — adds callbacks
- **CatBoost:** pass `catboost_mode=True` — passes `eval_set` as tuple

Test predictions are generated by averaging each fold's model's predictions on the test set. This gives 5 slightly different test predictions (each fold trains on 4/5 of the data) which average out fold-specific noise.

In [4]:
def run_cv(train_df, feature_cols, model_factory,
           test_df=None, lgbm_early_stopping=False, catboost_mode=False, label=''):
    """
    5-fold GroupKFold CV using pre-computed fold assignments.
    Applies target encodings inside each fold.
    Returns: (oof_auc, fold_aucs, oof_preds, test_preds_avg)
    test_preds_avg is None if test_df not provided.
    """
    n     = len(train_df)
    oof   = np.zeros(n)
    aucs  = []
    test_fold_preds = []
    t0    = time.time()

    for fold_idx in range(5):
        tr_mask  = train_df['fold'] != fold_idx
        val_mask = train_df['fold'] == fold_idx
        tr_idx   = np.where(tr_mask)[0]
        val_idx  = np.where(val_mask)[0]

        train_enc = apply_target_encodings(train_df.iloc[tr_idx], train_df.iloc[tr_idx])
        val_enc   = apply_target_encodings(train_df.iloc[tr_idx], train_df.iloc[val_idx])

        X_tr  = train_enc[feature_cols].to_numpy(dtype=np.float32)
        y_tr  = train_enc['PitNextLap'].to_numpy()
        X_val = val_enc[feature_cols].to_numpy(dtype=np.float32)
        y_val = val_enc['PitNextLap'].to_numpy()

        m = model_factory()

        if lgbm_early_stopping:
            callbacks = [lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
            m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=callbacks)
        elif catboost_mode:
            m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
        else:
            m.fit(X_tr, y_tr)

        oof[val_idx] = m.predict_proba(X_val)[:, 1]
        fold_auc = roc_auc_score(y_val, oof[val_idx])
        aucs.append(fold_auc)

        if test_df is not None:
            test_enc  = apply_target_encodings(train_df.iloc[tr_idx], test_df)
            X_test    = test_enc[feature_cols].to_numpy(dtype=np.float32)
            test_fold_preds.append(m.predict_proba(X_test)[:, 1])

        # Progress indicator
        if lgbm_early_stopping:
            extra = f'trees={m.best_iteration_}'
        elif hasattr(m, 'n_estimators'):
            extra = f'n_estimators={m.n_estimators}'
        else:
            extra = ''
        print(f'  Fold {fold_idx}: AUC={fold_auc:.4f}  {extra}')

    oof_auc = roc_auc_score(train_df['PitNextLap'], oof)
    elapsed = time.time() - t0
    print(f'  ─── OOF AUC: {oof_auc:.4f}  std={np.std(aucs):.4f}  n_feat={len(feature_cols)}  ({elapsed:.0f}s)')
    if label:
        print(f'  ─── {label}')

    test_avg = np.mean(test_fold_preds, axis=0) if test_fold_preds else None
    return oof_auc, np.array(aucs), oof, test_avg

print('run_cv defined.')

run_cv defined.


## 1 — LGBM Reference (same params as NB 10 Tier 4)

Reruns the NB 10 best configuration to get fresh OOF predictions for ρ comparison. Expected OOF AUC: ~0.8987.

In [5]:
LGBM_BEST = dict(
    n_estimators      = 2000,
    learning_rate     = 0.049228,
    num_leaves        = 62,
    min_child_samples = 91,
    subsample         = 0.752871,
    colsample_bytree  = 0.746388,
    reg_alpha         = 9.791678,
    reg_lambda        = 0.0,
    random_state      = 42,
    verbose           = -1,
)

print('LGBM reference: Optuna-tuned params from NB 05, 38-feature set.')
print('Expected OOF AUC: ~0.8987  (NB 10 Tier 4 result)\n')

auc_lgbm, aucs_lgbm, oof_lgbm, test_lgbm = run_cv(
    train, FEAT_COLS,
    model_factory=lambda: lgb.LGBMClassifier(**LGBM_BEST),
    test_df=test,
    lgbm_early_stopping=True,
    label='LGBM reference (NB 10 Tier 4 params)',
)

LGBM reference: Optuna-tuned params from NB 05, 38-feature set.
Expected OOF AUC: ~0.8987  (NB 10 Tier 4 result)

  Fold 0: AUC=0.9090  trees=286
  Fold 1: AUC=0.8731  trees=78
  Fold 2: AUC=0.9038  trees=92
  Fold 3: AUC=0.9142  trees=370
  Fold 4: AUC=0.8906  trees=174
  ─── OOF AUC: 0.8978  std=0.0148  n_feat=38  (41s)
  ─── LGBM reference (NB 10 Tier 4 params)


## 2 — Random Forest

Architecturally different from GBMs: bagging (train each tree on a bootstrap sample) + feature subsampling at each split. Predictions are averaged rather than residual-learned. Expected solo AUC: 0.85–0.87, but error distribution should differ meaningfully from LGBM.

Key params:
- `max_depth=20` — prevents deep overfitting while allowing complex interactions
- `min_samples_leaf=20` — smooths leaf predictions on 350k training rows  
- `max_features=0.4` — use 40% of features per split (≈15 features) — increases tree diversity

In [6]:
RF_PARAMS = dict(
    n_estimators    = 800,
    max_depth       = 20,
    min_samples_leaf= 20,
    max_features    = 0.4,
    n_jobs          = -1,
    random_state    = 42,
)

print('Random Forest CV — 800 trees, max_depth=20, max_features=0.4')
print('Note: RF is slow (~10–20 min on local CPU). Run on Kaggle P100 for faster results.\n')

auc_rf, aucs_rf, oof_rf, test_rf = run_cv(
    train, FEAT_COLS,
    model_factory=lambda: RandomForestClassifier(**RF_PARAMS),
    test_df=test,
    label='Random Forest',
)

delta_rf  = auc_rf - auc_lgbm
rho_rf, _ = spearmanr(oof_lgbm, oof_rf)
print(f'  Δ vs LGBM: {delta_rf:+.4f}  |  Spearman ρ vs LGBM: {rho_rf:.4f}  (target < 0.90)')

Random Forest CV — 800 trees, max_depth=20, max_features=0.4
Note: RF is slow (~10–20 min on local CPU). Run on Kaggle P100 for faster results.

  Fold 0: AUC=0.8948  n_estimators=800
  Fold 1: AUC=0.8404  n_estimators=800
  Fold 2: AUC=0.8690  n_estimators=800
  Fold 3: AUC=0.8992  n_estimators=800
  Fold 4: AUC=0.8793  n_estimators=800
  ─── OOF AUC: 0.8780  std=0.0210  n_feat=38  (1629s)
  ─── Random Forest
  Δ vs LGBM: -0.0199  |  Spearman ρ vs LGBM: 0.9464  (target < 0.90)


## 3 — Extra-Trees

Extreme randomisation of splits: instead of finding the optimal split threshold at each node, Extra-Trees draws the threshold randomly. This maximises variance reduction in the ensemble and typically produces lower correlation with GBMs than standard RF.

Key params:
- `max_depth=None` — unlimited depth (regularised by `min_samples_leaf` only)
- `min_samples_leaf=15` — slightly more aggressive than RF (more variance per tree, compensated by random splits)
- `max_features=0.3` — 30% features per split for maximum diversity

In [7]:
ET_PARAMS = dict(
    n_estimators    = 800,
    max_depth       = None,
    min_samples_leaf= 15,
    max_features    = 0.3,
    n_jobs          = -1,
    random_state    = 42,
)

print('Extra-Trees CV — 800 trees, unlimited depth, max_features=0.3\n')

auc_et, aucs_et, oof_et, test_et = run_cv(
    train, FEAT_COLS,
    model_factory=lambda: ExtraTreesClassifier(**ET_PARAMS),
    test_df=test,
    label='Extra-Trees',
)

delta_et  = auc_et - auc_lgbm
rho_et, _ = spearmanr(oof_lgbm, oof_et)
print(f'  Δ vs LGBM: {delta_et:+.4f}  |  Spearman ρ vs LGBM: {rho_et:.4f}  (target < 0.90)')

Extra-Trees CV — 800 trees, unlimited depth, max_features=0.3

  Fold 0: AUC=0.8865  n_estimators=800
  Fold 1: AUC=0.8586  n_estimators=800
  Fold 2: AUC=0.8731  n_estimators=800
  Fold 3: AUC=0.9011  n_estimators=800
  Fold 4: AUC=0.8715  n_estimators=800
  ─── OOF AUC: 0.8788  std=0.0145  n_feat=38  (420s)
  ─── Extra-Trees
  Δ vs LGBM: -0.0191  |  Spearman ρ vs LGBM: 0.9312  (target < 0.90)


## 4 — LightGBM DART

DART (Dropouts meet Multiple Additive Regression Trees): during training, a random fraction of previously-built trees are "dropped" (set to zero) before adding the next tree. This prevents any single tree from dominating and introduces structured noise similar to neural network dropout.

**Critical:** DART does not support early stopping — passing early_stopping callbacks raises an error or is silently ignored. Fixed `n_estimators=500`.

Params adapted from NB 10 LGBM base:
- `drop_rate=0.1` — 10% of trees dropped each round
- `skip_drop=0.5` — 50% chance to skip dropout entirely (prevents under-fitting)

In [8]:
LGBM_DART_PARAMS = dict(
    boosting_type   = 'dart',
    n_estimators    = 500,
    learning_rate   = 0.05,
    num_leaves      = 62,
    min_child_samples=91,
    subsample       = 0.75,
    colsample_bytree= 0.75,
    drop_rate       = 0.1,
    skip_drop       = 0.5,
    random_state    = 42,
    verbose         = -1,
)

print('LGBM-DART CV — 500 trees (no early stopping), drop_rate=0.1\n')

auc_dart, aucs_dart, oof_dart, test_dart = run_cv(
    train, FEAT_COLS,
    model_factory=lambda: lgb.LGBMClassifier(**LGBM_DART_PARAMS),
    test_df=test,
    lgbm_early_stopping=False,  # DART: no early stopping
    label='LightGBM DART',
)

delta_dart  = auc_dart - auc_lgbm
rho_dart, _ = spearmanr(oof_lgbm, oof_dart)
print(f'  Δ vs LGBM: {delta_dart:+.4f}  |  Spearman ρ vs LGBM: {rho_dart:.4f}  (target < 0.90)')

LGBM-DART CV — 500 trees (no early stopping), drop_rate=0.1

  Fold 0: AUC=0.9046  n_estimators=500
  Fold 1: AUC=0.8736  n_estimators=500
  Fold 2: AUC=0.8953  n_estimators=500
  Fold 3: AUC=0.9102  n_estimators=500
  Fold 4: AUC=0.8932  n_estimators=500
  ─── OOF AUC: 0.8953  std=0.0125  n_feat=38  (341s)
  ─── LightGBM DART
  Δ vs LGBM: -0.0026  |  Spearman ρ vs LGBM: 0.9726  (target < 0.90)


## 5 — CatBoost with `boosting_type='Plain'`

**Why prior CatBoost failed:** CatBoost's default `boosting_type='Ordered'` uses ordered target encoding — it computes statistics on permuted subsets of training data at each tree node. With GroupKFold-by-Race, consecutive laps from the same race appear in the same training fold, creating an ordered encoding that stops updating after 2–14 trees (essentially "memorises" the first few races).

**Fix:** `boosting_type='Plain'` switches to standard symmetric-tree GBDT without ordered encoding. Symmetric trees (all splits at a given depth use the same feature + threshold) are CatBoost's structural differentiator from LGBM's leaf-wise growth.

Driver and Race are passed as pre-computed numeric target encodings — no `cat_features`.

In [9]:
if not CATBOOST_AVAILABLE:
    print('CatBoost not available — skipping.')
    auc_cb, aucs_cb, oof_cb, test_cb = None, None, None, None
else:
    CB_PLAIN_PARAMS = dict(
        iterations    = 2000,
        learning_rate = 0.05,
        depth         = 8,
        boosting_type = 'Plain',
        eval_metric   = 'AUC',
        od_type       = 'Iter',
        od_wait       = 50,
        use_best_model= True,
        verbose       = 0,
        random_seed   = 42,
    )

    print('CatBoost-Plain CV — symmetric GBDT, depth=8, od_wait=50\n')

    auc_cb, aucs_cb, oof_cb, test_cb = run_cv(
        train, FEAT_COLS,
        model_factory=lambda: CatBoostClassifier(**CB_PLAIN_PARAMS),
        test_df=test,
        catboost_mode=True,
        label='CatBoost Plain',
    )

    delta_cb  = auc_cb - auc_lgbm
    rho_cb, _ = spearmanr(oof_lgbm, oof_cb)
    print(f'  Δ vs LGBM: {delta_cb:+.4f}  |  Spearman ρ vs LGBM: {rho_cb:.4f}  (target < 0.90)')

CatBoost-Plain CV — symmetric GBDT, depth=8, od_wait=50

  Fold 0: AUC=0.9081  
  Fold 1: AUC=0.8864  
  Fold 2: AUC=0.9047  
  Fold 3: AUC=0.9126  
  Fold 4: AUC=0.8945  
  ─── OOF AUC: 0.9016  std=0.0095  n_feat=38  (122s)
  ─── CatBoost Plain
  Δ vs LGBM: +0.0038  |  Spearman ρ vs LGBM: 0.9515  (target < 0.90)


## Correlation Analysis

Spearman ρ matrix for all model pairs. The key question: which models are diverse enough (ρ < 0.90) to contribute meaningful signal in Notebook 13's stacking ensemble?

**Interpretation guide:**
- ρ > 0.95: nearly identical error patterns — averaging will barely help
- 0.90 < ρ < 0.95: some diversity — rank averaging may help marginally
- ρ < 0.90: meaningful diversity — stacking can exploit complementary signals
- ρ < 0.85: strong diversity — stacking should clearly outperform solo models

In [10]:
# Build results dict — only include models that ran successfully
results = {'lgbm_ref': (auc_lgbm, aucs_lgbm, oof_lgbm)}
if oof_rf   is not None: results['rf']      = (auc_rf,   aucs_rf,   oof_rf)
if oof_et   is not None: results['et']      = (auc_et,   aucs_et,   oof_et)
if oof_dart is not None: results['dart']    = (auc_dart, aucs_dart, oof_dart)
if oof_cb   is not None: results['cb_plain']= (auc_cb,   aucs_cb,   oof_cb)

model_names = list(results.keys())

# ── OOF AUC summary ────────────────────────────────────────────────────────
print('OOF AUC Summary')
print('=' * 65)
print(f'  {"Model":<14} {"OOF AUC":>8}  {"Δ LGBM":>8}  Fold AUCs')
print('-' * 65)
for name, (auc, fold_aucs, _) in results.items():
    delta = auc - auc_lgbm if name != 'lgbm_ref' else 0.0
    delta_str = f'{delta:+.4f}' if name != 'lgbm_ref' else '  ref  '
    fold_str = '  '.join(f'{a:.4f}' for a in fold_aucs)
    print(f'  {name:<14} {auc:>8.4f}  {delta_str:>8}  {fold_str}')
print()

# ── Spearman ρ matrix ───────────────────────────────────────────────────────
print('Spearman ρ Matrix (OOF predictions)')
print('=' * 65)
header = '  ' + ''.join(f'{n:>10}' for n in model_names)
print(header)
for n1 in model_names:
    row = f'  {n1:<14}'
    for n2 in model_names:
        if n1 == n2:
            row += f'{"1.000":>10}'
        else:
            rho, _ = spearmanr(results[n1][2], results[n2][2])
            tag = ' *' if rho < 0.90 else ''
            row += f'{rho:.4f}{tag:>2}'.rjust(10)
    print(row)
print()
print('  * = below 0.90 diversity threshold — suitable for stacking')
print()

# ── Decision table ──────────────────────────────────────────────────────────
print('Stacking Inclusion Decision')
print('=' * 65)
print(f'  {"Model":<14} {"AUC":>8}  {"ρ vs LGBM":>10}  Verdict')
print('-' * 65)
for name, (auc, _, oof_pred) in results.items():
    if name == 'lgbm_ref':
        print(f'  {name:<14} {auc:>8.4f}  {"—":>10}  reference')
        continue
    rho, _ = spearmanr(oof_lgbm, oof_pred)
    verdict = 'INCLUDE (ρ<0.90)' if rho < 0.90 else ('BORDERLINE' if rho < 0.92 else 'EXCLUDE (too correlated)')
    print(f'  {name:<14} {auc:>8.4f}  {rho:>10.4f}  {verdict}')

OOF AUC Summary
  Model           OOF AUC    Δ LGBM  Fold AUCs
-----------------------------------------------------------------
  lgbm_ref         0.8978     ref    0.9090  0.8731  0.9038  0.9142  0.8906
  rf               0.8780   -0.0199  0.8948  0.8404  0.8690  0.8992  0.8793
  et               0.8788   -0.0191  0.8865  0.8586  0.8731  0.9011  0.8715
  dart             0.8953   -0.0026  0.9046  0.8736  0.8953  0.9102  0.8932
  cb_plain         0.9016   +0.0038  0.9081  0.8864  0.9047  0.9126  0.8945

Spearman ρ Matrix (OOF predictions)
    lgbm_ref        rf        et      dart  cb_plain
  lgbm_ref           1.000  0.9464    0.9312    0.9726    0.9515  
  rf              0.9464       1.000  0.9575    0.9488    0.8952 *
  et              0.9312    0.9575       1.000  0.9347    0.9094  
  dart            0.9726    0.9488    0.9347       1.000  0.9576  
  cb_plain        0.9515    0.8952 *  0.9094    0.9576       1.000

  * = below 0.90 diversity threshold — suitable for stacking

Sta

In [11]:
# ── Save OOF and test predictions ───────────────────────────────────────────

# OOF predictions (aligned to train row order)
oof_df = pd.DataFrame({'id': train['id'], 'fold': train['fold'], 'PitNextLap': train['PitNextLap']})
for name, (_, _, oof_pred) in results.items():
    oof_df[f'{name}_oof'] = oof_pred

oof_path = processed_dir / 'oof_predictions_nb11.parquet'
oof_df.to_parquet(oof_path, index=False)
print(f'Saved OOF: {oof_path}  shape={oof_df.shape}')

# Test predictions
test_preds_dict = {}
if test_lgbm is not None: test_preds_dict['lgbm_ref'] = test_lgbm
if test_rf   is not None: test_preds_dict['rf']       = test_rf
if test_et   is not None: test_preds_dict['et']       = test_et
if test_dart is not None: test_preds_dict['dart']     = test_dart
if test_cb   is not None: test_preds_dict['cb_plain'] = test_cb

test_df_out = pd.DataFrame({'id': test['id']})
for name, preds in test_preds_dict.items():
    test_df_out[f'{name}_pred'] = preds

test_path = processed_dir / 'test_predictions_nb11.parquet'
test_df_out.to_parquet(test_path, index=False)
print(f'Saved test: {test_path}  shape={test_df_out.shape}')

# Also save model summaries as pkl for NB 13
summary = {
    'oof_aucs':   {n: v[0] for n, v in results.items()},
    'fold_aucs':  {n: v[1] for n, v in results.items()},
    'lgbm_ref_auc': auc_lgbm,
    'feature_cols': FEAT_COLS,
    'spearman_rho': {n: float(spearmanr(oof_lgbm, results[n][2])[0])
                     for n in model_names if n != 'lgbm_ref'},
}
with open(models_dir / 'nb11_summary.pkl', 'wb') as f:
    pickle.dump(summary, f)
print(f'Saved summary: {models_dir / "nb11_summary.pkl"}')

Saved OOF: c:\Repos\predict-f1-pit-stops\data\processed\oof_predictions_nb11.parquet  shape=(439140, 8)
Saved test: c:\Repos\predict-f1-pit-stops\data\processed\test_predictions_nb11.parquet  shape=(188165, 6)
Saved summary: c:\Repos\predict-f1-pit-stops\models\nb11_summary.pkl


## Summary — Results and NB 13 Strategy

### OOF AUC Results

| Model | OOF AUC | Δ LGBM | ρ vs LGBM | Fold 1 AUC | Verdict |
|-------|---------|--------|-----------|------------|---------|
| lgbm_ref | 0.8978 | — | — | 0.8731 | reference |
| rf | 0.8780 | −0.0199 | 0.946 | 0.8404 | EXCLUDE |
| et | 0.8788 | −0.0191 | 0.931 | 0.8586 | EXCLUDE |
| dart | 0.8953 | −0.0026 | 0.973 | 0.8736 | EXCLUDE |
| **cb_plain** | **0.9016** | **+0.0038** | **0.952** | **0.8864** | **PRIMARY — new best model** |

### Key Findings

**1. CatBoost-Plain (0.9016) is the new best model — first algorithm to beat LGBM.**  
Symmetric-tree GBDT without ordered encoding, untuned default params. Projected LB: 0.9016 + 0.049 ≈ **0.951**. Remaining gap to top: ~0.004.

**2. No model broke ρ < 0.90 vs LGBM.**  
All models share the same 38 features and GroupKFold splits. The dominant features (Stint, prime_pit_window, TyreLife×compound) force all models toward the same decision boundaries regardless of architecture. Diversity requires different feature sets, not just different algorithms.  
The only pair below 0.90 in the full matrix: **RF × CatBoost (ρ = 0.895)** — but RF at 0.8780 is too weak to contribute positively to a stack.

**3. DART adds no diversity (ρ = 0.973 vs LGBM).**  
Tree dropout on the same feature set converges to the same prediction surface on 439k rows. Do not include in NB 13.

**4. ET is the most architecturally diverse vs LGBM (ρ = 0.931).**  
Also ρ = 0.909 vs CatBoost — borderline third model for a 3-way stack. Include only if metalearner gain is measurably positive.

**5. Fold 1 improved significantly with CatBoost (0.8864 vs 0.8731 for LGBM).**  
CatBoost's symmetric trees handle the distributional shift in fold 1's races better than LGBM's leaf-wise growth.

### Spearman ρ Matrix Summary (key pairs)

| Pair | ρ | Implication |
|------|---|-------------|
| LGBM × CatBoost | 0.952 | Correlated but CatBoost beats LGBM — worthwhile secondary |
| LGBM × ET | 0.931 | Most diverse LGBM pair — borderline third model |
| ET × CatBoost | 0.909 | Borderline diverse — include as optional third in NB 13 |
| RF × CatBoost | **0.895** | Only pair below 0.90 — RF too weak to use |
| DART × anything | ≥ 0.935 | No diversity value — drop from NB 13 |

### Notebook 13 Ensemble Plan

**Primary:** CatBoost-Plain (re-tuned in NB 12)  
**Secondary:** LGBM-ref (re-tuned in NB 12) — ρ = 0.952 vs CatBoost  
**Optional third:** ET — ρ = 0.909 vs CatBoost; include only if metalearner gain > +0.001  
**Drop:** RF (too weak), DART (no diversity)

Metalearner: `LogisticRegression(C=0.05)` on:
```python
X_meta = np.column_stack([
    cb_plain_platt, lgbm_platt,              # base model scores
    Stint_scaled, prime_pit_window_scaled,   # top raw features (V2 pattern)
    TyreLife_x_compound_ordinal_scaled, laps_to_driver_end_scaled,
])
```
Pre-check: if metalearner gain vs CatBoost solo < +0.001, submit CatBoost solo.

**Next:** Notebook 12 — Optuna re-tuning of CatBoost (primary, 100–200 trials) and LGBM (200 trials, multi-fold objective) on the 38-feature set. Target: CatBoost CV AUC > 0.906 → projected LB > 0.955.